In [1]:
from utils import getDevice #important to impot at the start for reproducibility :3 

Using device: xpu :3
Setting seed to 42:3


In [2]:
# GENERAL IMPORTS
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import time
import torch
import gc
import os
from IPython.display import clear_output

from torchvision.models import vgg16
from torchvision import models
from torchvision import transforms
from torchvision.ops import roi_pool
import torchvision
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split

# YOLO STUFF
from ultralytics import YOLO

In [ ]:
DEVICE = getDevice()
HEIGHT = 376
WIDTH = 1242
TARGET_TYPES = ['Car']
BATCH_SIZE = 8
NUM_WORKERS = 4

In [ ]:

class SimpleModel(nn.Module):
    def __init__(self, input_size: int = 2048):
        super().__init__()
        self.distance_regressor = nn.Sequential(
            nn.Linear(input_size, 2048),
            nn.ReLU(),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Linear(512, 1),
            nn.ReLU(),
            nn.Softplus()
        )
        self.feature_extractor = vgg16(weights=models.VGG16_Weights.DEFAULT).features

    def forward(self, stacked_images: torch.Tensor, boxes: torch.Tensor, verbose: bool = False) -> torch.Tensor:
        """
        Forward pass of the model. It takes a batch of images as input as well as boxes.

        Args: 
        - stacked_images: Tensor of shape (B,C,H,W) containing the batch of images.
        - boxes: Tensor of shape (N, 4) containing the bounding boxes.
        """
        #SINGLE BATCH IS PROCESSED AND THEN DELETED

        #FEATURE EXTRACTION--------------------------------------------------------------
        features = self.feature_extractor(stacked_images)
        if verbose:
            print(f"    1. Features shape: {features.shape}")

        #RESIZING THE BOXES--------------------------------------------------------------
        #I can't pass the boxes directly as the size of the image is different now
        H_in, W_in = HEIGHT, WIDTH
        H_out, W_out = features.shape[2], features.shape[3]
        scale_x = W_out / W_in
        scale_y = H_out / H_in
        scaled_boxes = []
        for box in boxes:
            scaled_box = box.clone()
            scaled_box[:, [0, 2]] *= scale_x  #Scale x1 and x2
            scaled_box[:, [1, 3]] *= scale_y  #Scale y1 and y2
            scaled_boxes.append(scaled_box)

        roi_boxes = []
        # I need to add indicies for the thing to work as expected with roi_pool
        for i, box in enumerate(scaled_boxes):
            indicies = torch.full((box.shape[0],), i, dtype=torch.float)#add batch index
            roi_boxes.append(torch.cat([indicies.unsqueeze(1), box], dim=1))

        #concatenate all final boxes that include batch index
        roi_boxes_tensor = torch.cat(roi_boxes, dim=0,).to(DEVICE)
        if verbose:
            print(f"    2. Roi shape of the boxes: {roi_boxes_tensor.shape}")

        #ROI POOLING----------------------------------------------------------------------
        poopoo= roi_pool(features,roi_boxes_tensor, output_size=(2,2))#I assumed this due to first layer in the net
        if verbose:
            print(f"    3. Pooled features shape: {poopoo.shape}")
        poopoo = poopoo.view(poopoo.size(0), -1)  #Flatten to achieve a clear feature vector

        #DISTANCE REGRESSION---------------------------------------------------------------
        distance = self.distance_regressor(poopoo)
        if verbose:
            print(f"    4.  Distance shape: {distance.shape}")

        #CLEANUP --------------------------------------------------------------------------
        del boxes, features,scaled_boxes, roi_boxes, roi_boxes_tensor, poopoo
        gc.collect()

        return distance

simpleModel = SimpleModel().to(DEVICE)

In [ ]:
def collate_fn(batch):
    images  = [item[0] for item in batch]   # list of PIL Images
    targets = [item[1] for item in batch]   # list of target dicts
    return images, targets

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((HEIGHT, WIDTH))
])

#DATASET IS LOADED AND DATALOADER IS CREATED

kitty_dataset =torchvision.datasets.Kitti(root="../datasets/", train=True,transform=transform, download=True)
kittty_dataloader = torch.utils.data.DataLoader(
    kitty_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_fn, 
    num_workers=NUM_WORKERS, 
    pin_memory=True
)
# kitty_dataset_test =torchvision.datasets.Kitti(root="../datasets/", train=False,transform=transform, download=True)
# kittty_dataloader_test = torch.utils.data.DataLoader(
#     kitty_dataset_test, 
#     batch_size=BATCH_SIZE, 
#     shuffle=False, 
#     collate_fn=collate_fn, 
#     num_workers=NUM_WORKERS, 
#     pin_memory=True
# )

KITTY_LENGTH = len(kitty_dataset)
BATCH_COUNT = KITTY_LENGTH // BATCH_SIZE + (KITTY_LENGTH % BATCH_SIZE > 0)
print(f"KITTY dataset is length: {KITTY_LENGTH}\nDataloader has length: {len(kittty_dataloader)}\nBatch count {BATCH_COUNT}")
#print(f"KITTY test dataset is length: {len(kitty_dataset_test)}\nDataloader has length: {len(kittty_dataloader_test)}")

#I try a single batch to analyze the data structure and types
images, targets_batch = next(iter(kittty_dataloader))
print(f"Type of the images data is: {type(images)} type of image {type(images[0])} len: {len(images)} and type of the target is {type(targets_batch)} of len {len(targets_batch)}")

KITTY dataset is length: 7481
Dataloader has length: 1871
Batch count 1871
KITTY test dataset is length: 7518
Dataloader has length: 1880
Type of the images data is: <class 'list'> type of image <class 'torch.Tensor'> len: 4 and type of the target is <class 'list'> of len 4


In [6]:
for i, targets in enumerate(targets_batch):
    print(f"Target type is {type(targets)} and length is {len(targets)}")
    for target in targets:
        print(type(target))  # Should be a dict
        obj_types  = target["type"]        # e.g. ['Car', 'Pedestrian']
        locations  = target["location"]    # Tensor [N, 3]: (X, Y, Z) in metres
        bboxes     = torch.Tensor(target["bbox"])        # Tensor [N, 4]: (x1, y1, x2, y2) pixels
        print(f"Object types: {obj_types} of type {type(obj_types)}")
        print(f"Locations: {locations} of type {type(locations)}")
        print(f"Bounding boxes: {bboxes} of type {type(bboxes)}")

    print("")

Target type is <class 'list'> and length is 5
<class 'dict'>
Object types: Car of type <class 'str'>
Locations: [-3.02, 1.62, 8.42] of type <class 'list'>
Bounding boxes: tensor([182.5700, 159.8900, 461.6800, 353.9600]) of type <class 'torch.Tensor'>
<class 'dict'>
Object types: Car of type <class 'str'>
Locations: [-2.9, 1.71, 16.29] of type <class 'list'>
Bounding boxes: tensor([423.0100, 185.6000, 531.9700, 257.8200]) of type <class 'torch.Tensor'>
<class 'dict'>
Object types: Car of type <class 'str'>
Locations: [2.2, 1.74, 33.12] of type <class 'list'>
Bounding boxes: tensor([638.2200, 178.1100, 681.5400, 212.8300]) of type <class 'torch.Tensor'>
<class 'dict'>
Object types: Van of type <class 'str'>
Locations: [-3.0, 1.69, 51.85] of type <class 'list'>
Bounding boxes: tensor([552.4400, 170.1500, 583.4700, 197.1700]) of type <class 'torch.Tensor'>
<class 'dict'>
Object types: DontCare of type <class 'str'>
Locations: [-1000.0, -1000.0, -1000.0] of type <class 'list'>
Bounding boxe

In [ ]:
def run_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, loss_fn, train:bool, epoch: str=""):
    """
    Runs an epoch. Can be used for training and evaluation of the model.

    Args:
        model: the model to train/evaluate
        loader: dataloader for the dataset
        optimizer: optimizer to use for training (ignored if train=False)
        error_fn: loss function to use for error calculation
        device: device to run on (cpu or cuda)
        train: whether to run in training mode (True) or evaluation mode (False)
        epoch: string that tells which epoch is currently running
    """
    model.train() if train else model.eval()
    model_loss = 0
    depths_true_all = []
    depths_pred_all = []
    counter = 0
    current_loss=0

    #set modes according to train state
    with torch.set_grad_enabled(train):
        for images_batch, targets_batch in loader:
            #preporcess the data, format it in a way that is expected by the model
            clear_output(wait=True)
            print(f"{epoch}; Batch {counter} in progress; ({counter/len(loader)*100:.2f}%); Loss: {current_loss:.4f}")

            # 1 IMAGES
            stacked_images = torch.stack(images_batch).to(DEVICE)
            # 2,3 BOXES and DEPTHS
            boxes =[]
            depths = []
            skip = []
            image_indicies = set(range(len(images_batch)))
            keep=[]
            for i, targets in enumerate(targets_batch):
                boxes_with_target = [torch.Tensor(target["bbox"]) for target in targets if target["type"] in TARGET_TYPES]
                depths_with_target = [target["location"][2] for target in targets if target["type"] in TARGET_TYPES]
                if len(boxes_with_target) > 0:
                    boxes.append(torch.stack(boxes_with_target))
                    depths.append(torch.Tensor(depths_with_target))
                else:
                    skip.append(i)

            # 3.5 check that there is actually data 
            if len(boxes) == 0:
                print("No boxes found in this batch, skipping...")
                del boxes, images_batch, targets_batch
                gc.collect()
                continue
            elif len(boxes)==1:
                #print("SINGLE IMAGE that contains cats")
                print("")

            #3.6 skip images that we do not want
            keep = list(image_indicies - set(skip))
            stacked_images = stacked_images[keep,:,:,:]

            # 4 FORWARD PASS: values are predicted and error calculation
            pred_depths = model(stacked_images, boxes)
            #print(f"Predicted depths type: {type(pred_depths)} and shape {pred_depths.shape}")
            #if len(boxes)>1:
            pred_depths = pred_depths.squeeze(-1).cpu()
            depths = torch.cat(depths).cpu()

            #print(f"Predicted depths type: {type(pred_depths)} and shape {pred_depths.shape}")
            #print(f"True depths type: {type(depths)} and shape {depths.shape}")

            loss = loss_fn(pred_depths, depths)
            current_loss = loss.item()

            depths_true_all.extend(depths.tolist())
            #if pred_depths.shape[0]==1:
                #depths_pred_all.append(pred_depths.item())
                #print("SINGLE PREDICTION APPENDED")
            #else:
            depths_pred_all.extend(pred_depths.flatten().tolist())
                #print(f"Multiple predictions appended: {pred_depths.shape[0]}")
            model_loss += loss.item()

            # 5 TRAINING STEP: training step perforned here 
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            del stacked_images, boxes
            #gc.collect()
            counter+=1

    gc.collect()            
    return model_loss / len(loader), depths_true_all, depths_pred_all

In [ ]:
def train(model: nn.Module, dataset: Dataset, optimizer: torch.optim.Optimizer, loss_fn, batch_size: int, epochs: int):
    predicted_depths, true_depths = [], []
    predicted_depths_test, true_depths_test = [], []

    #split dataset into train and validation sets
    val_size = int(0.2 * len(dataset))
    train_size = len(dataset) - val_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    #create dataloaders for both sets in the similar fashion
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True,
        collate_fn=collate_fn, 
        num_workers=NUM_WORKERS, 
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn, 
        num_workers=NUM_WORKERS, 
        pin_memory=True
    )

    for epoch in range(epochs):
        train_loss, true_d, pred_d = run_epoch(model, train_loader, optimizer, loss_fn, train=True, epoch=f"Training epoch {epoch}/{epochs}")
        test_loss, true_d_test, pred_d_test = run_epoch(model, val_loader, optimizer, loss_fn, train=False, epoch=f"Validation epoch {epoch}/{epochs}")
        #clear_output(wait=True)
        #print(f"Epoch {epoch+1}/{epochs}; Train Loss: {loss_fn(torch.Tensor(true_d), torch.Tensor(pred_d)):.4f}; Val Loss: {loss_fn(torch.Tensor(true_d_test), torch.Tensor(pred_d_test)):.4f}, expected: {train_loss}, {test_loss}")
        true_depths.extend(true_d)
        predicted_depths.extend(pred_d)
        true_depths_test.extend(true_d_test)
        predicted_depths_test.extend(pred_d_test)
        
    return true_depths, predicted_depths, true_depths_test, predicted_depths_test

In [ ]:
# Create the loss function,  model
loss_fn = nn.L1Loss() # MAE loss is same as L1Loss
model_vgg16_zhu = SimpleModel().to(DEVICE)

# Create the optimizer
optimizer = torch.optim.Adam(model_vgg16_zhu.parameters(), lr=0.001, eps=1.0)
# Epcohs number to train
epochs = 10
dataloader = kittty_dataloader


In [10]:
train_loss, true_d, pred_d = run_epoch(model_vgg16_zhu, dataloader, optimizer, loss_fn, train=True, epoch=f"Initial run")

Batch 1870 in progress; (99.95%)
No boxes found in this batch, skipping...


In [11]:
true_d_np = torch.Tensor(true_d)
pred_d_np = torch.Tensor(pred_d)
print(f"Loss after 1 epoch is {loss_fn(true_d_np, pred_d_np):.4f}, expected: {train_loss}")
print(f"True depths shape: {len(true_d)}, Predicted depths shape: {len(pred_d)} and train loss {train_loss:.4f}")
pred_d_np = torch.Tensor(pred_d)

Loss after 1 epoch is 5.5295, expected: 5.596663770638705
True depths shape: 28742, Predicted depths shape: 28742 and train loss 5.5967
